# 02 - Sliding-window chunking

Sliding-window chunking is fixed-size chunking with overlap. Instead of jumping directly from one chunk to the next, each new chunk reuses some words from the previous chunk.

In RAG, this helps when the answer is near a chunk boundary. The cost is duplication: you index more text and may retrieve similar chunks.

In [ ]:
from pathlib import Path
import re

from pypdf import PdfReader

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/sample_documents/AUG242026_05B2203_AAO.pdf").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome_RAG checkout")
PDF_PATH = ROOT / "data" / "sample_documents" / "AUG242026_05B2203_AAO.pdf"

def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_pdf_pages(path):
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = normalize_text(page.extract_text() or "")
        if text:
            pages.append({"page": page_number, "text": text})
    return pages

def chunk_stats(chunks):
    lengths = [len(chunk["text"].split()) for chunk in chunks]
    return {"chunks": len(chunks), "min_words": min(lengths), "avg_words": round(sum(lengths) / len(lengths), 1), "max_words": max(lengths)}

def print_chunks(chunks, limit=5, preview_chars=650):
    for chunk in chunks[:limit]:
        print(f"{chunk['id']} | page {chunk['page_start']}-{chunk['page_end']} | words {chunk['start_word']}-{chunk['end_word']}")
        print(chunk["text"][:preview_chars])
        print("-" * 100)
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")

pages = load_pdf_pages(PDF_PATH)
{"pages": len(pages), "words": sum(len(page["text"].split()) for page in pages)}

## Sliding-window chunking function

The key idea is `step = size - overlap`.

If `size=180` and `overlap=45`, then each chunk has up to 180 words, but the next chunk starts only 135 words later. That means 45 words are repeated.

This repeated area protects evidence near boundaries.

In [ ]:
def sliding_window_chunks(pages, size=180, overlap=45):
    """Split each page into fixed-size chunks that overlap with the previous chunk."""
    if size <= 0:
        raise ValueError("size must be positive")
    if not 0 <= overlap < size:
        raise ValueError("overlap must be at least 0 and smaller than size")

    chunks = []
    step = size - overlap

    for page in pages:
        words = page["text"].split()

        for start in range(0, len(words), step):
            end = min(start + size, len(words))
            chunks.append({
                "id": f"window-{len(chunks) + 1}",
                "page_start": page["page"],
                "page_end": page["page"],
                "start_word": start,
                "end_word": end,
                "text": " ".join(words[start:end]),
            })

            if end == len(words):
                break

    return chunks

chunks = sliding_window_chunks(pages, size=180, overlap=45)
chunk_stats(chunks)

In [ ]:
print_chunks(chunks, limit=6)

## How to judge it

Sliding-window chunking is better than fixed-size chunking when important evidence is often split between two chunks.

It is worse when overlap creates too many near-duplicate chunks. In production RAG, overlap improves recall but can increase index size, retrieval noise, and prompt cost.